# Enabling Voices Systematic Review - Round 1: Filtering
## LLM-Assisted Screening for AI + Dementia/Aphasia + Communication Research

**Purpose:** Filter 478 papers to identify those meeting all three criteria:
1. Population: Dementia and/or Aphasia focus
2. Technology: Artificial Intelligence involvement
3. Focus: Language and/or Interaction

**Model:** Llama 3.1 8B Instruct (recommended for classification tasks)

**Platform:** SDU UCloud

**Inputs:** PDF files of research articles  
**Outputs:** 
- Excel file with screening decisions and confidence scores
- Separate sheets for Include/Exclude/Manual Review
- Validation sample for inter-rater reliability

## Cell 1: Installation

Run this cell first to install required packages.

In [ ]:
# Install required packages
!pip install transformers torch accelerate pypdf2 pandas openpyxl bitsandbytes -q

print("✓ Packages installed")

## Cell 2: Imports and GPU Check

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import PyPDF2
import pandas as pd
import json
import os
import re
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ WARNING: No GPU detected. Processing will be slow.")

## Cell 3: Configuration

**IMPORTANT:** Update these paths for your UCloud environment.

In [ ]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS FOR YOUR UCLOUD SETUP
# =============================================================================

# Input/Output paths
PDF_FOLDER = "/work/PDFs"  # UPDATE: Path to your PDF folder on UCloud
OUTPUT_DIR = "/work/outputs"  # UPDATE: Output directory
OUTPUT_PREFIX = "enabling_voices_round1"  # Prefix for output files

# Model configuration
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MAX_TOKENS_PER_ARTICLE = 6000  # Increased for full-text analysis
TEMPERATURE = 0.1  # Low temperature for consistent classification

# Processing options
TEST_MODE_LIMIT = None  # Set to e.g., 10 for testing, None for full processing
CHECKPOINT_EVERY = 10  # Save checkpoint every N papers

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Derived paths
OUTPUT_FILE = f"{OUTPUT_DIR}/{OUTPUT_PREFIX}_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
CHECKPOINT_FILE = f"{OUTPUT_DIR}/{OUTPUT_PREFIX}_checkpoint.json"

print(f"PDF Folder: {PDF_FOLDER}")
print(f"Output File: {OUTPUT_FILE}")
print(f"Max tokens per article: {MAX_TOKENS_PER_ARTICLE}")

## Cell 4: HuggingFace Login

**IMPORTANT:** Replace with your HuggingFace token. You need access to Llama 3.1.

In [ ]:
from huggingface_hub import login

# Replace with your HuggingFace token
# Get one at: https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_TOKEN_HERE"  # UPDATE THIS

login(token=HF_TOKEN)
print("✓ Logged in to HuggingFace")

## Cell 5: Load Model

This will take 2-5 minutes depending on your GPU.

In [ ]:
# 4-bit quantization for memory efficiency
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading Llama 3.1 8B Instruct... This may take 2-5 minutes.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("✓ Model loaded successfully!")
if torch.cuda.is_available():
    print(f"  GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Cell 6: PDF Text Extraction

In [ ]:
def extract_text_from_pdf(pdf_path, max_pages=None):
    """
    Extract text from PDF file.
    
    Args:
        pdf_path: Path to PDF file
        max_pages: Maximum pages to extract (None = all pages)
    
    Returns:
        Extracted text string or None if extraction fails
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            total_pages = len(pdf_reader.pages)
            pages_to_read = total_pages if max_pages is None else min(max_pages, total_pages)
            
            for page_num in range(pages_to_read):
                page = pdf_reader.pages[page_num]
                page_text = page.extract_text() or ""
                text += page_text + "\n"
            
            # Clean up text
            text = text.replace('\x00', '')  # Remove null bytes
            text = re.sub(r'\n{3,}', '\n\n', text)  # Reduce excessive newlines
            text = re.sub(r' {2,}', ' ', text)  # Reduce excessive spaces
            
            return text, total_pages
    except Exception as e:
        print(f"  Error extracting text from {pdf_path}: {e}")
        return None, 0

print("✓ PDF extraction function loaded")

## Cell 7: Round 1 Filtering Prompt

This prompt is designed for classification/filtering decisions.

In [ ]:
FILTERING_PROMPT = '''You are screening research papers for a systematic review on AI technologies that SUPPORT communication for people with dementia and/or aphasia.

CRITICAL DISTINCTION:
This review focuses on AI that HELPS people communicate (assistive/supportive), NOT AI that DETECTS or DIAGNOSES conditions through speech/language analysis.

SCREENING CRITERIA - A paper should be INCLUDED if it meets ALL THREE:
1. POPULATION: Involves people with dementia OR aphasia (including primary progressive aphasia, post-stroke aphasia, or communication difficulties in dementia)
2. TECHNOLOGY: Involves artificial intelligence (machine learning, deep learning, NLP, LLMs, chatbots, computer vision, speech recognition, intelligent/adaptive systems, social robots with AI capabilities)
3. COMMUNICATION SUPPORT: The AI is used to SUPPORT, ASSIST, ENABLE, or ENHANCE communication, language, speech, conversation, or social interaction for the target population

EXCLUSION CRITERIA (do NOT include):
- DETECTION/DIAGNOSIS STUDIES: AI used to detect, diagnose, screen for, or assess dementia or aphasia through speech, language, or cognitive markers (e.g., "detecting MCI from speech patterns", "automatic aphasia severity assessment", "ML classification of dementia subtypes")
- Digital tools WITHOUT AI (e.g., simple video calling, photo albums, basic reminder apps)
- Studies only on healthy older adults or other populations without dementia/aphasia
- Pure theoretical/ethical discussions without technology implementation or evaluation
- Studies on caregivers only without patient involvement
- Non-intelligent assistive devices
- Outcome measurement tools that only ASSESS communication but don\'t SUPPORT it

INCLUDE EXAMPLES (AI that SUPPORTS communication):
- Social robots that facilitate conversation or social interaction
- Chatbots or virtual agents that help users communicate or practice language
- AI-powered AAC (augmentative and alternative communication) devices
- Speech recognition systems that assist with communication
- NLP-based tools that simplify or augment language for users
- AI systems that prompt, cue, or scaffold conversation
- Intelligent reminiscence systems that support meaningful interaction
- Adaptive communication interfaces

EXCLUDE EXAMPLES (AI that DETECTS/DIAGNOSES):
- ML models that classify dementia vs healthy from speech features
- Automatic detection of aphasia severity from language samples
- AI screening tools for cognitive impairment
- Speech biomarker analysis for early dementia detection
- NLP analysis to predict disease progression
- Diagnostic decision support systems (unless they also include communication support)

ARTICLE TEXT:
{article_text}

Analyze this article and return ONLY a JSON object with your screening decision:

{{
  "screening_decision": {{
    "include": true/false,
    "confidence": 1-5,
    "decision_rationale": "Brief explanation (max 100 chars)"
  }},
  "population_assessment": {{
    "has_dementia_focus": true/false/unclear,
    "has_aphasia_focus": true/false/unclear,
    "population_details": "Brief description (max 100 chars)",
    "population_confidence": 1-5
  }},
  "technology_assessment": {{
    "has_ai_technology": true/false/unclear,
    "ai_type": ["list", "of", "AI", "types"],
    "technology_details": "Brief description (max 100 chars)",
    "technology_confidence": 1-5,
    "is_detection_study": true/false
  }},
  "communication_assessment": {{
    "has_communication_support": true/false/unclear,
    "communication_type": ["verbal", "nonverbal", "social_interaction"],
    "communication_details": "Brief description (max 100 chars)",
    "communication_confidence": 1-5,
    "support_vs_detect": "support/detect/both/unclear"
  }},
  "study_metadata": {{
    "study_type": "empirical/review/theoretical/development/other",
    "publication_type": "journal_article/conference/thesis/book_chapter/other"
  }},
  "manual_review_flag": {{
    "needs_review": true/false,
    "review_reason": "Why manual review needed (if applicable)"
  }},
  "key_quote": "One sentence from text supporting AI involvement (max 150 chars)"
}}

CONFIDENCE SCALE:
1 = Very uncertain, limited information
2 = Somewhat uncertain
3 = Moderately confident
4 = Confident
5 = Very confident, clear evidence

AI_TYPE OPTIONS: speech_recognition, NLP, machine_learning, deep_learning, computer_vision, chatbot, social_robot_AI, emotion_detection, LLM, intelligent_agent, adaptive_system, AAC, other_AI, none, unclear

SUPPORT_VS_DETECT:
- "support" = AI helps users communicate (INCLUDE)
- "detect" = AI detects/diagnoses condition from speech/language (EXCLUDE)
- "both" = Paper includes both aspects (flag for MANUAL REVIEW)
- "unclear" = Cannot determine (flag for MANUAL REVIEW)

Return ONLY the JSON object, no other text.\'\'\' 

print("✓ Filtering prompt loaded (with detection exclusion)")
print(f"  Prompt length: {len(FILTERING_PROMPT)} characters")

## Cell 8: JSON Extraction Function

In [ ]:
def extract_first_json(text):
    """
    Extract the FIRST complete JSON object from text.
    Uses brace-counting to handle cases where LLM outputs multiple JSON objects.
    """
    if not text:
        return None
    
    # Find the first opening brace
    start_idx = text.find('{')
    if start_idx == -1:
        return None
    
    # Track brace depth to find matching closing brace
    depth = 0
    in_string = False
    escape_next = False
    
    for i, char in enumerate(text[start_idx:], start=start_idx):
        if escape_next:
            escape_next = False
            continue
        
        if char == '\\':
            escape_next = True
            continue
        
        if char == '"' and not escape_next:
            in_string = not in_string
            continue
        
        if in_string:
            continue
        
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                # Found the complete first JSON object
                json_str = text[start_idx:i+1]
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    # Try to clean common issues
                    json_str = json_str.replace('True', 'true').replace('False', 'false')
                    json_str = json_str.replace("'", '"')
                    json_str = re.sub(r',\s*}', '}', json_str)  # Remove trailing commas
                    json_str = re.sub(r',\s*]', ']', json_str)
                    try:
                        return json.loads(json_str)
                    except:
                        return None
    
    return None

print("✓ JSON extraction function loaded")

## Cell 9: Annotation Function

In [ ]:
def screen_article(pdf_path, model, tokenizer, max_tokens=6000):
    """
    Screen a single article and return structured filtering decision.
    
    Args:
        pdf_path: Path to PDF file
        model: Loaded LLM model
        tokenizer: Model tokenizer
        max_tokens: Maximum tokens to include from article
    
    Returns:
        Dictionary with screening results
    """
    filename = pdf_path.name if hasattr(pdf_path, 'name') else str(pdf_path)
    result = {
        '_filename': filename,
        '_pdf_path': str(pdf_path)
    }
    
    # Extract text
    text, total_pages = extract_text_from_pdf(pdf_path)
    result['_total_pages'] = total_pages
    
    if not text:
        result['_processing_status'] = 'failed_text_extraction'
        result['_error'] = 'Could not extract text from PDF'
        return result
    
    result['_extracted_chars'] = len(text)
    
    # Truncate text if needed
    tokens = tokenizer.encode(text)
    original_tokens = len(tokens)
    if len(tokens) > max_tokens:
        text = tokenizer.decode(tokens[:max_tokens])
        result['_truncated'] = True
        result['_original_tokens'] = original_tokens
    else:
        result['_truncated'] = False
    
    result['_tokens_used'] = min(len(tokens), max_tokens)
    
    # Create prompt
    prompt = FILTERING_PROMPT.format(article_text=text)
    
    # Format for chat
    messages = [
        {"role": "system", "content": "You are a research assistant screening academic articles for a systematic review. Return ONLY valid JSON, no other text."},
        {"role": "user", "content": prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=TEMPERATURE,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Extract response (only the generated part)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Extract JSON
    screening = extract_first_json(response)
    
    if screening:
        result['_raw_screening'] = screening
        
        # Flatten key decision fields for easy filtering
        sd = screening.get('screening_decision', {})
        result['include'] = sd.get('include', None)
        result['confidence'] = sd.get('confidence', None)
        result['decision_rationale'] = sd.get('decision_rationale', '')
        
        # Population
        pa = screening.get('population_assessment', {})
        result['has_dementia'] = pa.get('has_dementia_focus', None)
        result['has_aphasia'] = pa.get('has_aphasia_focus', None)
        result['population_details'] = pa.get('population_details', '')
        
        # Technology
        ta = screening.get('technology_assessment', {})
        result['has_ai'] = ta.get('has_ai_technology', None)
        result['ai_types'] = ','.join(ta.get('ai_type', [])) if isinstance(ta.get('ai_type'), list) else str(ta.get('ai_type', ''))
        result['technology_details'] = ta.get('technology_details', '')
        result['is_detection_study'] = ta.get('is_detection_study', None)  # NEW FIELD
        
        # Communication
        ca = screening.get('communication_assessment', {})
        result['has_communication_support'] = ca.get('has_communication_support', None)  # RENAMED
        result['communication_types'] = ','.join(ca.get('communication_type', [])) if isinstance(ca.get('communication_type'), list) else str(ca.get('communication_type', ''))
        result['support_vs_detect'] = ca.get('support_vs_detect', '')  # NEW FIELD
        
        # Study metadata
        sm = screening.get('study_metadata', {})
        result['study_type'] = sm.get('study_type', '')
        result['publication_type'] = sm.get('publication_type', '')
        
        # Manual review flag
        mrf = screening.get('manual_review_flag', {})
        result['needs_manual_review'] = mrf.get('needs_review', False)
        result['review_reason'] = mrf.get('review_reason', '')
        
        # Key quote
        result['key_quote'] = screening.get('key_quote', '')
        
        result['_processing_status'] = 'success'
    else:
        result['_processing_status'] = 'failed_json_extraction'
        result['_error'] = 'Could not extract valid JSON from response'
        result['_raw_response_preview'] = response[:500] if response else 'No response'
    
    return result

print("✓ Screening function loaded (with detection exclusion fields)")

## Cell 10: Results Saving Functions

In [ ]:
def save_results_to_excel(results, output_path):
    """
    Save results to Excel file with multiple sheets for different categories.
    """
    if not results:
        print("No results to save.")
        return None
    
    # Convert to DataFrame, excluding nested dictionaries
    flat_results = []
    for r in results:
        flat = {k: v for k, v in r.items() if not isinstance(v, dict)}
        flat_results.append(flat)
    
    df_all = pd.DataFrame(flat_results)
    
    # Create category columns for filtering
    df_all['_category'] = 'unknown'
    
    # Categorize based on screening decision
    mask_success = df_all['_processing_status'] == 'success'
    mask_include = df_all['include'] == True
    mask_exclude = df_all['include'] == False
    mask_manual = df_all['needs_manual_review'] == True
    mask_unclear_ai = df_all['has_ai'].isin(['unclear', 'Unclear'])
    
    df_all.loc[mask_success & mask_include, '_category'] = 'include'
    df_all.loc[mask_success & mask_exclude & ~mask_manual, '_category'] = 'exclude'
    df_all.loc[mask_success & mask_manual, '_category'] = 'manual_review'
    df_all.loc[mask_success & mask_unclear_ai, '_category'] = 'unclear_ai_review'
    df_all.loc[~mask_success, '_category'] = 'processing_failed'
    
    # Create separate DataFrames
    df_include = df_all[df_all['_category'] == 'include'].copy()
    df_exclude = df_all[df_all['_category'] == 'exclude'].copy()
    df_manual = df_all[df_all['_category'].isin(['manual_review', 'unclear_ai_review'])].copy()
    df_failed = df_all[df_all['_category'] == 'processing_failed'].copy()
    
    # Select columns for output
    key_cols = ['_filename', 'include', 'confidence', 'decision_rationale',
                'has_dementia', 'has_aphasia', 'has_ai', 'ai_types',
                'is_detection_study', 'has_communication_support', 'support_vs_detect',
                'study_type', 'needs_manual_review', 'review_reason', 'key_quote', '_category']
    
    meta_cols = ['_processing_status', '_total_pages', '_tokens_used', '_truncated']
    
    # Save to Excel with multiple sheets
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # Summary sheet first
        summary_data = {
            'Category': ['Total Processed', 'Include', 'Exclude', 'Manual Review Needed', 'Processing Failed'],
            'Count': [len(df_all), len(df_include), len(df_exclude), len(df_manual), len(df_failed)]
        }
        pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)
        
        # All results
        df_all.to_excel(writer, sheet_name='All Results', index=False)
        
        # Category sheets
        if len(df_include) > 0:
            df_include[key_cols + meta_cols].to_excel(writer, sheet_name='Include', index=False)
        if len(df_exclude) > 0:
            df_exclude[key_cols + meta_cols].to_excel(writer, sheet_name='Exclude', index=False)
        if len(df_manual) > 0:
            df_manual[key_cols + meta_cols].to_excel(writer, sheet_name='Manual Review', index=False)
        if len(df_failed) > 0:
            df_failed.to_excel(writer, sheet_name='Failed', index=False)
    
    print(f"\n✓ Saved to {output_path}")
    print(f"  Summary:")
    print(f"    - Include: {len(df_include)}")
    print(f"    - Exclude: {len(df_exclude)}")
    print(f"    - Manual Review: {len(df_manual)}")
    print(f"    - Failed: {len(df_failed)}")
    
    return df_all

print("✓ Save functions loaded")

## Cell 11: Validation Sample Creator

In [ ]:
def create_validation_sample(results, sample_size=50, stratified=True):
    """
    Create a stratified validation sample for inter-rater reliability testing.
    
    Args:
        results: List of screening results
        sample_size: Number of papers to include in validation sample
        stratified: If True, sample proportionally from each category
    
    Returns:
        List of selected results for validation
    """
    import random
    
    # Filter successful results
    successful = [r for r in results if r.get('_processing_status') == 'success']
    
    if len(successful) <= sample_size:
        return successful
    
    if stratified:
        # Group by decision
        include = [r for r in successful if r.get('include') == True]
        exclude = [r for r in successful if r.get('include') == False]
        unclear = [r for r in successful if r.get('has_ai') in ['unclear', 'Unclear']]
        
        # Calculate proportional sample sizes
        total = len(successful)
        n_include = max(1, int(sample_size * len(include) / total))
        n_exclude = max(1, int(sample_size * len(exclude) / total))
        n_unclear = max(1, int(sample_size * len(unclear) / total))
        
        # Adjust to hit target
        remaining = sample_size - n_include - n_exclude - n_unclear
        n_include += remaining
        
        # Sample from each category
        sample = []
        sample.extend(random.sample(include, min(n_include, len(include))))
        sample.extend(random.sample(exclude, min(n_exclude, len(exclude))))
        sample.extend(random.sample(unclear, min(n_unclear, len(unclear))))
        
        return sample
    else:
        return random.sample(successful, sample_size)

print("✓ Validation sample function loaded")

## Cell 12: Test Single Article

Run this to verify the setup is working before processing all papers.

In [ ]:
# Find PDF files
pdf_files = list(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files in {PDF_FOLDER}")

if not pdf_files:
    print("\n⚠ No PDF files found! Please check the PDF_FOLDER path.")
else:
    print(f"\nTesting with: {pdf_files[0].name}")
    print("="*60)
    
    test_result = screen_article(pdf_files[0], model, tokenizer, MAX_TOKENS_PER_ARTICLE)
    
    print(f"\nProcessing Status: {test_result.get('_processing_status')}")
    print(f"Pages: {test_result.get('_total_pages')}")
    print(f"Tokens used: {test_result.get('_tokens_used')}")
    print(f"Truncated: {test_result.get('_truncated')}")
    
    if test_result.get('_processing_status') == 'success':
        print(f"\n--- SCREENING DECISION ---")
        print(f"Include: {test_result.get('include')}")
        print(f"Confidence: {test_result.get('confidence')}/5")
        print(f"Rationale: {test_result.get('decision_rationale')}")
        print(f"\n--- CRITERIA ASSESSMENT ---")
        print(f"Dementia focus: {test_result.get('has_dementia')}")
        print(f"Aphasia focus: {test_result.get('has_aphasia')}")
        print(f"AI technology: {test_result.get('has_ai')}")
        print(f"AI types: {test_result.get('ai_types')}")
        print(f"Communication focus: {test_result.get('has_communication')}")
        print(f"\n--- KEY QUOTE ---")
        print(f"{test_result.get('key_quote')}")
        print(f"\n✓ Test successful!")
    else:
        print(f"\nError: {test_result.get('_error', 'Unknown')}")
        if '_raw_response_preview' in test_result:
            print(f"\nResponse preview:\n{test_result['_raw_response_preview']}")

## Cell 13: Main Processing Loop

This will process all 478 papers. Estimated time: 2-4 hours depending on GPU.

In [ ]:
# Get PDF files
pdf_files = list(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files")

# Apply test mode limit if set
if TEST_MODE_LIMIT:
    pdf_files = pdf_files[:TEST_MODE_LIMIT]
    print(f"🧪 TEST MODE: Processing only {TEST_MODE_LIMIT} articles")

# Load checkpoint if exists
results = []
processed_files = set()

if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            results = checkpoint.get('results', [])
            processed_files = set(checkpoint.get('processed_files', []))
        print(f"✓ Loaded checkpoint: {len(processed_files)} files already processed")
    except:
        print("⚠ Could not load checkpoint, starting fresh")

# Track statistics
stats = {'success': 0, 'include': 0, 'exclude': 0, 'failed': 0}

# Process each PDF
start_time = datetime.now()
for i, pdf_path in enumerate(tqdm(pdf_files, desc="Screening articles")):
    filename = pdf_path.name
    
    # Skip if already processed
    if filename in processed_files:
        continue
    
    # Screen article
    result = screen_article(pdf_path, model, tokenizer, MAX_TOKENS_PER_ARTICLE)
    results.append(result)
    processed_files.add(filename)
    
    # Update statistics
    if result.get('_processing_status') == 'success':
        stats['success'] += 1
        if result.get('include'):
            stats['include'] += 1
        else:
            stats['exclude'] += 1
    else:
        stats['failed'] += 1
    
    # Progress update every 10 files
    if (i + 1) % 10 == 0:
        elapsed = (datetime.now() - start_time).total_seconds() / 60
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        remaining = (len(pdf_files) - i - 1) / rate if rate > 0 else 0
        print(f"\n  Progress: {i+1}/{len(pdf_files)} | Include: {stats['include']} | Exclude: {stats['exclude']} | ~{remaining:.0f} min remaining")
    
    # Save checkpoint
    if len(processed_files) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({
                'results': results,
                'processed_files': list(processed_files),
                'stats': stats
            }, f)

print(f"\n" + "="*60)
print(f"✓ PROCESSING COMPLETE!")
print(f"  Total: {len(results)}")
print(f"  Include: {stats['include']}")
print(f"  Exclude: {stats['exclude']}")
print(f"  Failed: {stats['failed']}")
print(f"  Time: {(datetime.now() - start_time).total_seconds() / 60:.1f} minutes")

## Cell 14: Save Final Results

In [ ]:
# Save all results to Excel
df_all = save_results_to_excel(results, OUTPUT_FILE)

# Print detailed summary
print("\n" + "="*60)
print("SCREENING SUMMARY")
print("="*60)

if df_all is not None and len(df_all) > 0:
    print(f"\nBy Category:")
    print(df_all['_category'].value_counts().to_string())
    
    print(f"\nAI Technology Types (in included papers):")
    included = df_all[df_all['_category'] == 'include']
    if len(included) > 0:
        ai_types = included['ai_types'].str.split(',').explode().str.strip()
        print(ai_types.value_counts().head(10).to_string())
    
    print(f"\nStudy Types:")
    print(df_all['study_type'].value_counts().to_string())

## Cell 15: Create Validation Sample (10-20%)

This creates a stratified sample for human annotation to compare with LLM decisions.

In [ ]:
# Create validation sample (approximately 15% of total)
validation_size = max(50, int(len(results) * 0.15))  # At least 50, or 15%
validation_sample = create_validation_sample(results, sample_size=validation_size, stratified=True)

print(f"\nValidation sample: {len(validation_sample)} papers ({len(validation_sample)/len(results)*100:.1f}%)")

# Save validation sample
validation_file = OUTPUT_FILE.replace('.xlsx', '_validation_sample.xlsx')
save_results_to_excel(validation_sample, validation_file)

# Also save as CSV for easy manual coding
validation_csv = OUTPUT_FILE.replace('.xlsx', '_validation_sample.csv')
df_validation = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, dict)} for r in validation_sample])

# Add columns for human coding
df_validation['human_include'] = ''  # For human coder to fill
df_validation['human_confidence'] = ''  # For human coder to fill
df_validation['human_notes'] = ''  # For human coder comments

df_validation.to_csv(validation_csv, index=False)

print(f"\n✓ Validation sample saved:")
print(f"  Excel: {validation_file}")
print(f"  CSV (for manual coding): {validation_csv}")
print(f"\nNext step: Have team members manually code the validation sample,")
print(f"then compare with LLM decisions to assess inter-rater reliability.")

## Cell 16: Cleanup and Final Report

In [ ]:
# Remove checkpoint file
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print("✓ Checkpoint file removed")

print("\n" + "="*60)
print("ROUND 1 SCREENING COMPLETE!")
print("="*60)
print(f"\nOutput files created:")
print(f"  1. Main results: {OUTPUT_FILE}")
print(f"  2. Validation sample (Excel): {validation_file}")
print(f"  3. Validation sample (CSV): {validation_csv}")

print(f"\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("""
1. VALIDATION (before proceeding to Round 2):
   a. Have 2+ team members manually code the validation sample
   b. Calculate inter-rater reliability (Cohen's kappa)
   c. If kappa < 0.7, review disagreements and refine criteria

2. MODEL COMPARISON (optional but recommended):
   a. Run Qwen2.5-7B on the same validation sample
   b. Compare both models against human coders
   c. Select best-performing model for Round 2

3. ROUND 2 PREPARATION:
   a. Review 'Manual Review' sheet for edge cases
   b. Finalize inclusion list
   c. Prepare Round 2 extraction prompt for included papers
""")

## Cell 17: Quick Inter-Rater Reliability Calculator (Optional)

Run this AFTER human coding is complete to calculate agreement.

In [ ]:
# This cell calculates inter-rater reliability AFTER human coding
# Uncomment and run after filling in human_include column in validation CSV

"""
from sklearn.metrics import cohen_kappa_score, accuracy_score

# Load the coded validation sample
coded_file = validation_csv  # Update if saved elsewhere
df_coded = pd.read_csv(coded_file)

# Filter to rows where human coded
df_coded = df_coded[df_coded['human_include'].notna() & (df_coded['human_include'] != '')]

if len(df_coded) > 0:
    # Convert to binary
    llm_decisions = df_coded['include'].map({True: 1, False: 0, 'True': 1, 'False': 0})
    human_decisions = df_coded['human_include'].map({True: 1, False: 0, 'True': 1, 'False': 0, 
                                                      'yes': 1, 'no': 0, 'Yes': 1, 'No': 0,
                                                      1: 1, 0: 0, '1': 1, '0': 0})
    
    # Calculate metrics
    kappa = cohen_kappa_score(llm_decisions, human_decisions)
    accuracy = accuracy_score(llm_decisions, human_decisions)
    agreement = (llm_decisions == human_decisions).mean() * 100
    
    print("INTER-RATER RELIABILITY RESULTS")
    print("="*40)
    print(f"Sample size: {len(df_coded)}")
    print(f"Raw agreement: {agreement:.1f}%")
    print(f"Cohen's Kappa: {kappa:.3f}")
    print(f"Accuracy: {accuracy:.3f}")
    print()
    print("Kappa interpretation:")
    print("  < 0.20: Poor")
    print("  0.21-0.40: Fair")
    print("  0.41-0.60: Moderate")
    print("  0.61-0.80: Substantial")
    print("  0.81-1.00: Almost perfect")
    
    # Show disagreements
    disagreements = df_coded[llm_decisions != human_decisions]
    if len(disagreements) > 0:
        print(f"\nDisagreements ({len(disagreements)} papers):")
        print(disagreements[['_filename', 'include', 'human_include', 'decision_rationale']].to_string())
else:
    print("No human-coded data found. Please fill in the 'human_include' column.")
"""